# ronin-code-1.5b — FREE training on Colab (T4, $0)

Trains Ronin's QLoRA adapter on the free Colab T4. **Before running:** `Runtime → Change runtime type → T4 GPU`.

Honesty rails: the frozen test split is hash-verified before training; the eval never invents a score;
the ship gate is pre-committed — **valid_tool_json 4/4 AND >60/91** on the frozen 91-case eval, or archive.

Free-tier reality (plan around it): sessions time out (~hours) and can disconnect at any moment —
the trainer checkpoints every 100 steps, so add `--resume` to Cell 3 and re-run after a disconnect.
T4 availability is not guaranteed. Fallback: **Kaggle** (30 GPU-hrs/week free) — same cells verbatim.


In [ ]:
# Cell 1 — clone + pinned deps (a few minutes)
!git clone https://github.com/rohithkandula19/Ronin.git /content/Ronin 2>/dev/null || (cd /content/Ronin && git pull)
%cd /content/Ronin
!pip install -q -r training/cuda/requirements.txt
!pip install -q -e packages/dialect -e training
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > T4!')


In [ ]:
# Cell 2 — deterministic corpus (post D1/D2 fix) + frozen-split verification
!python -m ronin_training.dataset_builder --volumes docs/volumes --out training/data/generated
!python -m ronin_training.synthetic_corpus --target 5000 --seed 42 --out training/data/synthetic
!mkdir -p training/data/merged
!for s in train valid; do cat training/data/generated/$s.jsonl training/data/synthetic/$s.jsonl > training/data/merged/$s.jsonl; done
!cp training/data/synthetic/test.jsonl training/data/merged/test.jsonl
!cp training/data/synthetic/test.jsonl.sha256 training/data/merged/test.jsonl.sha256
# preflight: config + FROZEN SPLIT hash — refuses to train on a tampered split
!python training/cuda/train_cuda.py --check --qlora --fp16 --data training/data/merged


In [ ]:
# Cell 2b — PREFLIGHT the phase-12 adapter config, before you spend a single GPU minute.
#
# Cell 3 below trains the *v1* ronin-code-1.5b recipe (lr 2e-4, seq 2048). The
# phase-12 adapter is a different experiment: lr 1e-5, seq 4096, grad checkpointing,
# and — the load-bearing one — a holdout split BY TASK rather than by example.
# Splitting by example puts two examples of the same task on both sides, so the score
# you report is partly a memorisation score and nothing about the run looks wrong.
# This cell refuses that config by name.
#
# No torch, no GPU: a mistake here costs seconds. On a free T4 under a 12-hour
# ceiling, finding a config error 40 minutes in costs the session.

CONFIG = 'training/config/adapter_sft.yaml'   # or adapter_dpo.yaml for pass 2
ROWS   = 'training/data/adapter/sft.jsonl'    # ronin_training.harvest output
OUT    = 'training/data/adapter/split'

# Validate the config on its own first — this needs no corpus at all.
!python -m ronin_training.adapter validate $CONFIG

# Then preflight against the real rows and write the split.
!python -m ronin_training.adapter preflight --config $CONFIG --rows $ROWS --out $OUT

# 0 = every check passed and the split is written.
# 1 = a check failed; each line of the report says what to change.
# 2 = the config itself is malformed.

In [ ]:
# Cell 3 — QLoRA train on the T4 (fp16 + 4-bit nf4, ckpt every 100 steps)
# After a disconnect: re-run Cells 1-2, then add --resume here.
#
# TWO RECIPES LIVE HERE. Without --config this trains the v1 ronin-code-1.5b
# constants (lr 2e-4, seq 2048, seed 42). With --config it trains what the YAML says
# (lr 1e-5, seq 4096, seed 0) — a 20x difference on the single most consequential
# number. Run --check first: it prints which recipe would run, and the run report in
# training/reports/ records it, so two runs can actually be compared afterwards.

# --- the phase-12 adapter (what Cell 2b just preflighted) ---
!python training/cuda/train_cuda.py --check --config $CONFIG --data $OUT
!python training/cuda/train_cuda.py --config $CONFIG --qlora --fp16 --save-steps 100 \
    --data $OUT --out training/adapters/ronin-adapter-qwen1.5b/sft

# --- or the v1 recipe, unchanged and still reproducible ---
# !python training/cuda/train_cuda.py --qlora --fp16 --save-steps 100 \
#     --data training/data/merged --out training/adapters/ronin-code-1.5b-v5

In [ ]:
# Cell 4 — make the artifact survive the session: Google Drive + the free-HF upload command
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ronin-adapters
!cp -r training/adapters/ronin-code-1.5b-v5/adapters /content/drive/MyDrive/ronin-adapters/ronin-code-1.5b-v5
print('Adapter saved to Drive: MyDrive/ronin-adapters/ronin-code-1.5b-v5')
print()
print('Publish with a FREE Hugging Face account (only after the eval clears the gate):')
print('  pip install -U huggingface_hub && huggingface-cli login')
print('  hf upload <your-hf-user>/ronin-code-1.5b training/adapters/ronin-code-1.5b-v5/adapters --repo-type model')


In [ ]:
# Cell 5 — the honest eval: frozen 91-case set, baseline delta, provenance-stamped report
# Scores land in training/reports/ with model+adapter+commit SHA. NO score is ever invented.
!python -m ronin_training.eval_runner \
    --provider hf --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --adapter training/adapters/ronin-code-1.5b-v5/adapters --baseline
print('Gate: ship ONLY if valid_tool_json 4/4 AND >60/91. Miss it -> archive, honestly.')


In [ ]:
# Cell 5b — the three-way comparison: base qwen vs this adapter vs a hosted model.
#
# One number is not a result. 'the adapter scored 41%' means nothing without the base
# model's score on the same tasks at the same seeds — the adapter's whole claim is a
# DELTA over base on tool-syntax validity and recovery, not an absolute.
#
# If the adapter does not beat base, that IS the finding. Publish it. An honest
# negative result is a better signal to a reader than a vague positive one.

# The $0 lane: local weights, no API key, no network at inference time.
!cp examples/models.local.toml .ronin/models.toml

# 1. base model — leave RONIN_ADAPTER unset, and this is the honest baseline.
!PYTHONPATH=src python -m ronin eval --model ronin-qwen-local \
    --regression-gate --json base.json --markdown base.md

# 2. the adapter you just trained — same tasks, same suite.
!PYTHONPATH=src RONIN_ADAPTER=training/adapters/ronin-adapter-qwen1.5b/sft \
    python -m ronin eval --model ronin-qwen-local \
    --regression-gate --json adapter.json --markdown adapter.md

# 3. paired against a hosted model, if you have a key for one. Same per-task seeds.
# !PYTHONPATH=src python -m ronin duel --model ronin-qwen-local --model kimi-k2 \
#     --seed 7 --markdown duel.md

# See what would run without loading anything at all:
!PYTHONPATH=src python -m ronin eval --dry-run --regression-gate

# ronin_training.adapter.threeway assembles base/adapter/hosted into one table. It is
# a library, not a CLI — import assemble_report() and render_markdown() from a script.
#
# NO NUMBER BELONGS IN THIS NOTEBOOK UNTIL A RUN PRODUCES IT. A score you are reading
# here that you did not generate is a placeholder, and it is wrong.